In [15]:
import pandas as pd 
import numpy as np

all_methods = ['kt_gaussian', 'iid', 'kt_matern', 'kt_sobolev', 'kt_inverse_multiquadric', 'kt_predictions', 'stein_thinning', 'arfpy', 'influence']

# Figure 8

In [12]:
results_aggregated = pd.read_csv("../package_metadata/results_aggregation.csv")
all_methods = ['iid', 'kt_gaussian', 'stein_thinning', 'arfpy', 'influence']
results_aggregated = results_aggregated[results_aggregated['method_total'].isin(all_methods)]

metrics_to_analyze = [
    "mae_mean",
    "top_k_mean",
    "mmd_mean",
    "explanation_time_mean", 
    "mae_std",
    "top_k_std",
    "mmd_std",
    "explanation_time_std", 
]

baseline_method = "iid"

results_with_improvements = results_aggregated.copy()

group_cols = ['dataset_name', 'model_name', 'explainer_total', 'size', 'task']

for group_name, group in results_aggregated.groupby(group_cols):
    dataset_name, model_name, explainer_total, size, task = group_name

    baseline_row = group[group['method_total'] == baseline_method]
    if baseline_row.empty:
        continue

    baseline_idx = baseline_row.index[0]

    # ---------- SET BASELINE REDUCTIONS TO 0 ----------
    for metric in metrics_to_analyze:
        if metric in results_aggregated.columns:
            results_with_improvements.loc[
                baseline_idx, f'{metric}_pct_reduction'
            ] = 0.0

    # ---------- OTHER METHODS ----------
    for method in all_methods:
        if method == baseline_method:
            continue

        method_rows = group[group['method_total'] == method]
        if method_rows.empty:
            continue

        for idx in method_rows.index:
            for metric in metrics_to_analyze:
                if metric not in results_aggregated.columns:
                    continue

                baseline_val = baseline_row[metric].values[0]
                method_val = results_aggregated.loc[idx, metric]

                if (
                    pd.isna(baseline_val)
                    or pd.isna(method_val)
                    or baseline_val == 0
                ):
                    results_with_improvements.loc[
                        idx, f'{metric}_pct_reduction'
                    ] = np.nan
                    continue

                pct_reduction = (
                    (baseline_val - method_val)
                    / baseline_val
                ) * 100

                results_with_improvements.loc[
                    idx, f'{metric}_pct_reduction'
                ] = pct_reduction


In [13]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ================== CONFIG ==================
output_root = "Image8"
palette = "GnBu"

metrics_to_analyze = [
    "mae_mean",
    "top_k_mean",
    "mmd_mean",
    "explanation_time_mean",
    "mae_std",
    "top_k_std",
    "mmd_std",
    "explanation_time_std",
]

# ================== MEDIAN ANNOTATION ==================
def annotate_medians(ax, data, x, y, hue, palette):
    medians = (
        data
        .groupby([x, hue])[y]
        .median()
        .reset_index()
    )

    xticks = ax.get_xticks()
    xlabels = [t.get_text() for t in ax.get_xticklabels()]
    hue_levels = sorted(data[hue].dropna().unique())

    n_hue = len(hue_levels)
    width = 0.8
    step = width / n_hue

    colors = sns.color_palette(palette, n_hue)

    for _, row in medians.iterrows():
        x_pos = xlabels.index(row[x])
        hue_pos = hue_levels.index(row[hue])

        xpos = (
            xticks[x_pos]
            - width / 2
            + step / 2
            + hue_pos * step
        )

        r, g, b = colors[hue_pos]
        luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
        text_color = "white" if luminance < 0.5 else "black"

        ax.text(
            xpos,
            row[y],
            f"{row[y]:.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=text_color
        )

# ================== MAIN LOOP ==================
all_explainers = results_with_improvements["explainer_total"].unique()

os.makedirs(output_root, exist_ok=True)

for metric in metrics_to_analyze:
    metric_col = f"{metric}_pct_reduction"
    if metric_col not in results_with_improvements.columns:
        continue

    metric_dir = os.path.join(output_root, metric)
    os.makedirs(metric_dir, exist_ok=True)

    # ---- robust y-limits per metric ----
    q_low = results_with_improvements[metric_col].quantile(0.02)
    q_high = results_with_improvements[metric_col].quantile(0.98)
    padding = 0.05 * (q_high - q_low)

    y_min = q_low - padding
    y_max = 110

    for explainer in all_explainers:
        fig, axes = plt.subplots(2, 2, figsize=(20, 14))
        fig.suptitle(
            f"{metric} — Explainer: {explainer}",
            fontsize=18,
            fontweight="bold",
            y=1.02
        )

        for ax in axes.flat:
            ax.set_ylim(y_min, y_max)
            ax.axhline(0, linestyle="--", color="red", alpha=0.4)

        # ---------- NN - Regression ----------
        ax1 = axes[0, 0]
        data_nn_reg = results_with_improvements[
            (results_with_improvements["model_name"] == "nn") &
            (results_with_improvements["task"] == "regression") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_nn_reg.empty:
            sns.boxplot(
                data=data_nn_reg,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax1,
                palette=palette
            )
            annotate_medians(
                ax1, data_nn_reg,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax1.set_title("NN - Regression")
        ax1.set_ylabel(f"{metric} % reduction")
        ax1.tick_params(axis="x", rotation=45)
        ax1.legend(title="Compression")

        # ---------- NN - Classification ----------
        ax2 = axes[0, 1]
        data_nn_clf = results_with_improvements[
            (results_with_improvements["model_name"] == "nn") &
            (results_with_improvements["task"] == "classification") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_nn_clf.empty:
            sns.boxplot(
                data=data_nn_clf,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax2,
                palette=palette
            )
            annotate_medians(
                ax2, data_nn_clf,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax2.set_title("NN - Classification")
        ax2.set_ylabel("")
        ax2.tick_params(axis="x", rotation=45)
        ax2.legend(title="Compression")

        # ---------- XGBoost - Regression ----------
        ax3 = axes[1, 0]
        data_xgb_reg = results_with_improvements[
            (results_with_improvements["model_name"] == "xgboost") &
            (results_with_improvements["task"] == "regression") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_xgb_reg.empty:
            sns.boxplot(
                data=data_xgb_reg,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax3,
                palette=palette
            )
            annotate_medians(
                ax3, data_xgb_reg,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax3.set_title("XGBoost - Regression")
        ax3.set_xlabel("Method")
        ax3.set_ylabel(f"{metric} % reduction")
        ax3.tick_params(axis="x", rotation=45)
        ax3.legend(title="Compression")

        # ---------- XGBoost - Classification ----------
        ax4 = axes[1, 1]
        data_xgb_clf = results_with_improvements[
            (results_with_improvements["model_name"] == "xgboost") &
            (results_with_improvements["task"] == "classification") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_xgb_clf.empty:
            sns.boxplot(
                data=data_xgb_clf,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax4,
                palette=palette
            )
            annotate_medians(
                ax4, data_xgb_clf,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax4.set_title("XGBoost - Classification")
        ax4.set_xlabel("Method")
        ax4.set_ylabel("")
        ax4.tick_params(axis="x", rotation=45)
        ax4.legend(title="Compression")

        plt.tight_layout()

        fname = f"{explainer}.png"
        plt.savefig(os.path.join(metric_dir, fname), dpi=300, bbox_inches="tight")
        plt.close()


/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/3118177948.py:186: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax3.legend(title="Compression")
/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/3118177948.py:216: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax4.legend(title="Compression")
/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/3118177948.py:186: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax3.legend(title="Compression")
/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/3118177948.py:216: UserWarning: No artists with labels found to put in 

# Figure 7

In [8]:
results_aggregated = pd.read_csv("../package_metadata/results_aggregation.csv")
all_methods = ['iid', 'kt_gaussian', 'kt_predictions']
results_aggregated = results_aggregated[results_aggregated['method_total'].isin(all_methods)]

metrics_to_analyze = [
    "mae_mean",
    "top_k_mean",
    "mmd_mean",
    "explanation_time_mean", 
    "mae_std",
    "top_k_std",
    "mmd_std",
    "explanation_time_std", 
]

baseline_method = "iid"

results_with_improvements = results_aggregated.copy()

group_cols = ['dataset_name', 'model_name', 'explainer_total', 'size', 'task']

for group_name, group in results_aggregated.groupby(group_cols):
    dataset_name, model_name, explainer_total, size, task = group_name

    baseline_row = group[group['method_total'] == baseline_method]
    if baseline_row.empty:
        continue

    baseline_idx = baseline_row.index[0]

    # ---------- SET BASELINE REDUCTIONS TO 0 ----------
    for metric in metrics_to_analyze:
        if metric in results_aggregated.columns:
            results_with_improvements.loc[
                baseline_idx, f'{metric}_pct_reduction'
            ] = 0.0

    # ---------- OTHER METHODS ----------
    for method in all_methods:
        if method == baseline_method:
            continue

        method_rows = group[group['method_total'] == method]
        if method_rows.empty:
            continue

        for idx in method_rows.index:
            for metric in metrics_to_analyze:
                if metric not in results_aggregated.columns:
                    continue

                baseline_val = baseline_row[metric].values[0]
                method_val = results_aggregated.loc[idx, metric]

                if (
                    pd.isna(baseline_val)
                    or pd.isna(method_val)
                    or baseline_val == 0
                ):
                    results_with_improvements.loc[
                        idx, f'{metric}_pct_reduction'
                    ] = np.nan
                    continue

                pct_reduction = (
                    (baseline_val - method_val)
                    / baseline_val
                ) * 100

                results_with_improvements.loc[
                    idx, f'{metric}_pct_reduction'
                ] = pct_reduction


In [9]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ================== CONFIG ==================
output_root = "Image7"
palette = "GnBu"

metrics_to_analyze = [
    "mae_mean",
    "top_k_mean",
    "mmd_mean",
    "explanation_time_mean",
    "mae_std",
    "top_k_std",
    "mmd_std",
    "explanation_time_std",
]

# ================== MEDIAN ANNOTATION ==================
def annotate_medians(ax, data, x, y, hue, palette):
    medians = (
        data
        .groupby([x, hue])[y]
        .median()
        .reset_index()
    )

    xticks = ax.get_xticks()
    xlabels = [t.get_text() for t in ax.get_xticklabels()]
    hue_levels = sorted(data[hue].dropna().unique())

    n_hue = len(hue_levels)
    width = 0.8
    step = width / n_hue

    colors = sns.color_palette(palette, n_hue)

    for _, row in medians.iterrows():
        x_pos = xlabels.index(row[x])
        hue_pos = hue_levels.index(row[hue])

        xpos = (
            xticks[x_pos]
            - width / 2
            + step / 2
            + hue_pos * step
        )

        r, g, b = colors[hue_pos]
        luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
        text_color = "white" if luminance < 0.5 else "black"

        ax.text(
            xpos,
            row[y],
            f"{row[y]:.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=text_color
        )

# ================== MAIN LOOP ==================
all_explainers = results_with_improvements["explainer_total"].unique()

os.makedirs(output_root, exist_ok=True)

for metric in metrics_to_analyze:
    metric_col = f"{metric}_pct_reduction"
    if metric_col not in results_with_improvements.columns:
        continue

    metric_dir = os.path.join(output_root, metric)
    os.makedirs(metric_dir, exist_ok=True)

    # ---- robust y-limits per metric ----
    q_low = results_with_improvements[metric_col].quantile(0.02)
    q_high = results_with_improvements[metric_col].quantile(0.98)
    padding = 0.05 * (q_high - q_low)

    y_min = q_low - padding
    y_max = 110

    for explainer in all_explainers:
        fig, axes = plt.subplots(2, 2, figsize=(20, 14))
        fig.suptitle(
            f"{metric} — Explainer: {explainer}",
            fontsize=18,
            fontweight="bold",
            y=1.02
        )

        for ax in axes.flat:
            ax.set_ylim(y_min, y_max)
            ax.axhline(0, linestyle="--", color="red", alpha=0.4)

        # ---------- NN - Regression ----------
        ax1 = axes[0, 0]
        data_nn_reg = results_with_improvements[
            (results_with_improvements["model_name"] == "nn") &
            (results_with_improvements["task"] == "regression") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_nn_reg.empty:
            sns.boxplot(
                data=data_nn_reg,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax1,
                palette=palette
            )
            annotate_medians(
                ax1, data_nn_reg,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax1.set_title("NN - Regression")
        ax1.set_ylabel(f"{metric} % reduction")
        ax1.tick_params(axis="x", rotation=45)
        ax1.legend(title="Compression")

        # ---------- NN - Classification ----------
        ax2 = axes[0, 1]
        data_nn_clf = results_with_improvements[
            (results_with_improvements["model_name"] == "nn") &
            (results_with_improvements["task"] == "classification") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_nn_clf.empty:
            sns.boxplot(
                data=data_nn_clf,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax2,
                palette=palette
            )
            annotate_medians(
                ax2, data_nn_clf,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax2.set_title("NN - Classification")
        ax2.set_ylabel("")
        ax2.tick_params(axis="x", rotation=45)
        ax2.legend(title="Compression")

        # ---------- XGBoost - Regression ----------
        ax3 = axes[1, 0]
        data_xgb_reg = results_with_improvements[
            (results_with_improvements["model_name"] == "xgboost") &
            (results_with_improvements["task"] == "regression") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_xgb_reg.empty:
            sns.boxplot(
                data=data_xgb_reg,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax3,
                palette=palette
            )
            annotate_medians(
                ax3, data_xgb_reg,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax3.set_title("XGBoost - Regression")
        ax3.set_xlabel("Method")
        ax3.set_ylabel(f"{metric} % reduction")
        ax3.tick_params(axis="x", rotation=45)
        ax3.legend(title="Compression")

        # ---------- XGBoost - Classification ----------
        ax4 = axes[1, 1]
        data_xgb_clf = results_with_improvements[
            (results_with_improvements["model_name"] == "xgboost") &
            (results_with_improvements["task"] == "classification") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_xgb_clf.empty:
            sns.boxplot(
                data=data_xgb_clf,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax4,
                palette=palette
            )
            annotate_medians(
                ax4, data_xgb_clf,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax4.set_title("XGBoost - Classification")
        ax4.set_xlabel("Method")
        ax4.set_ylabel("")
        ax4.tick_params(axis="x", rotation=45)
        ax4.legend(title="Compression")

        plt.tight_layout()

        fname = f"{explainer}.png"
        plt.savefig(os.path.join(metric_dir, fname), dpi=300, bbox_inches="tight")
        plt.close()


/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/310678964.py:186: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax3.legend(title="Compression")
/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/310678964.py:216: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax4.legend(title="Compression")
/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/310678964.py:186: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax3.legend(title="Compression")
/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/310678964.py:216: UserWarning: No artists with labels found to put in lege

# Figure 6

In [10]:
results_aggregated = pd.read_csv("../package_metadata/results_aggregation.csv")
all_methods = ['iid', 'kt_gaussian', 'kt_matern', 'kt_inverse_multiquadric']
results_aggregated = results_aggregated[results_aggregated['method_total'].isin(all_methods)]

metrics_to_analyze = [
    "mae_mean",
    "top_k_mean",
    "mmd_mean",
    "explanation_time_mean", 
    "mae_std",
    "top_k_std",
    "mmd_std",
    "explanation_time_std", 
]

baseline_method = "iid"

results_with_improvements = results_aggregated.copy()

group_cols = ['dataset_name', 'model_name', 'explainer_total', 'size', 'task']

for group_name, group in results_aggregated.groupby(group_cols):
    dataset_name, model_name, explainer_total, size, task = group_name

    baseline_row = group[group['method_total'] == baseline_method]
    if baseline_row.empty:
        continue

    baseline_idx = baseline_row.index[0]

    # ---------- SET BASELINE REDUCTIONS TO 0 ----------
    for metric in metrics_to_analyze:
        if metric in results_aggregated.columns:
            results_with_improvements.loc[
                baseline_idx, f'{metric}_pct_reduction'
            ] = 0.0

    # ---------- OTHER METHODS ----------
    for method in all_methods:
        if method == baseline_method:
            continue

        method_rows = group[group['method_total'] == method]
        if method_rows.empty:
            continue

        for idx in method_rows.index:
            for metric in metrics_to_analyze:
                if metric not in results_aggregated.columns:
                    continue

                baseline_val = baseline_row[metric].values[0]
                method_val = results_aggregated.loc[idx, metric]

                if (
                    pd.isna(baseline_val)
                    or pd.isna(method_val)
                    or baseline_val == 0
                ):
                    results_with_improvements.loc[
                        idx, f'{metric}_pct_reduction'
                    ] = np.nan
                    continue

                pct_reduction = (
                    (baseline_val - method_val)
                    / baseline_val
                ) * 100

                results_with_improvements.loc[
                    idx, f'{metric}_pct_reduction'
                ] = pct_reduction


In [11]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ================== CONFIG ==================
output_root = "Image6"
palette = "GnBu"

metrics_to_analyze = [
    "mae_mean",
    "top_k_mean",
    "mmd_mean",
    "explanation_time_mean",
    "mae_std",
    "top_k_std",
    "mmd_std",
    "explanation_time_std",
]

# ================== MEDIAN ANNOTATION ==================
def annotate_medians(ax, data, x, y, hue, palette):
    medians = (
        data
        .groupby([x, hue])[y]
        .median()
        .reset_index()
    )

    xticks = ax.get_xticks()
    xlabels = [t.get_text() for t in ax.get_xticklabels()]
    hue_levels = sorted(data[hue].dropna().unique())

    n_hue = len(hue_levels)
    width = 0.8
    step = width / n_hue

    colors = sns.color_palette(palette, n_hue)

    for _, row in medians.iterrows():
        x_pos = xlabels.index(row[x])
        hue_pos = hue_levels.index(row[hue])

        xpos = (
            xticks[x_pos]
            - width / 2
            + step / 2
            + hue_pos * step
        )

        r, g, b = colors[hue_pos]
        luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
        text_color = "white" if luminance < 0.5 else "black"

        ax.text(
            xpos,
            row[y],
            f"{row[y]:.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=text_color
        )

# ================== MAIN LOOP ==================
all_explainers = results_with_improvements["explainer_total"].unique()

os.makedirs(output_root, exist_ok=True)

for metric in metrics_to_analyze:
    metric_col = f"{metric}_pct_reduction"
    if metric_col not in results_with_improvements.columns:
        continue

    metric_dir = os.path.join(output_root, metric)
    os.makedirs(metric_dir, exist_ok=True)

    # ---- robust y-limits per metric ----
    q_low = results_with_improvements[metric_col].quantile(0.02)
    q_high = results_with_improvements[metric_col].quantile(0.98)
    padding = 0.05 * (q_high - q_low)

    y_min = q_low - padding
    y_max = 110

    for explainer in all_explainers:
        fig, axes = plt.subplots(2, 2, figsize=(20, 14))
        fig.suptitle(
            f"{metric} — Explainer: {explainer}",
            fontsize=18,
            fontweight="bold",
            y=1.02
        )

        for ax in axes.flat:
            ax.set_ylim(y_min, y_max)
            ax.axhline(0, linestyle="--", color="red", alpha=0.4)

        # ---------- NN - Regression ----------
        ax1 = axes[0, 0]
        data_nn_reg = results_with_improvements[
            (results_with_improvements["model_name"] == "nn") &
            (results_with_improvements["task"] == "regression") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_nn_reg.empty:
            sns.boxplot(
                data=data_nn_reg,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax1,
                palette=palette
            )
            annotate_medians(
                ax1, data_nn_reg,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax1.set_title("NN - Regression")
        ax1.set_ylabel(f"{metric} % reduction")
        ax1.tick_params(axis="x", rotation=45)
        ax1.legend(title="Compression")

        # ---------- NN - Classification ----------
        ax2 = axes[0, 1]
        data_nn_clf = results_with_improvements[
            (results_with_improvements["model_name"] == "nn") &
            (results_with_improvements["task"] == "classification") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_nn_clf.empty:
            sns.boxplot(
                data=data_nn_clf,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax2,
                palette=palette
            )
            annotate_medians(
                ax2, data_nn_clf,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax2.set_title("NN - Classification")
        ax2.set_ylabel("")
        ax2.tick_params(axis="x", rotation=45)
        ax2.legend(title="Compression")

        # ---------- XGBoost - Regression ----------
        ax3 = axes[1, 0]
        data_xgb_reg = results_with_improvements[
            (results_with_improvements["model_name"] == "xgboost") &
            (results_with_improvements["task"] == "regression") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_xgb_reg.empty:
            sns.boxplot(
                data=data_xgb_reg,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax3,
                palette=palette
            )
            annotate_medians(
                ax3, data_xgb_reg,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax3.set_title("XGBoost - Regression")
        ax3.set_xlabel("Method")
        ax3.set_ylabel(f"{metric} % reduction")
        ax3.tick_params(axis="x", rotation=45)
        ax3.legend(title="Compression")

        # ---------- XGBoost - Classification ----------
        ax4 = axes[1, 1]
        data_xgb_clf = results_with_improvements[
            (results_with_improvements["model_name"] == "xgboost") &
            (results_with_improvements["task"] == "classification") &
            (results_with_improvements["method_total"].isin(all_methods)) &
            (results_with_improvements["explainer_total"] == explainer)
        ]

        if not data_xgb_clf.empty:
            sns.boxplot(
                data=data_xgb_clf,
                x="method_total",
                y=metric_col,
                hue="compression_coefficient",
                ax=ax4,
                palette=palette
            )
            annotate_medians(
                ax4, data_xgb_clf,
                "method_total", metric_col,
                "compression_coefficient", palette
            )

        ax4.set_title("XGBoost - Classification")
        ax4.set_xlabel("Method")
        ax4.set_ylabel("")
        ax4.tick_params(axis="x", rotation=45)
        ax4.legend(title="Compression")

        plt.tight_layout()

        fname = f"{explainer}.png"
        plt.savefig(os.path.join(metric_dir, fname), dpi=300, bbox_inches="tight")
        plt.close()


/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/2114671232.py:186: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax3.legend(title="Compression")
/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/2114671232.py:216: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax4.legend(title="Compression")
/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/2114671232.py:186: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax3.legend(title="Compression")
/var/folders/rv/f71wkbsd1qs63ff8371lyjx40000gn/T/ipykernel_41104/2114671232.py:216: UserWarning: No artists with labels found to put in 